# San Diego County Income Distribution

This notebook uses the 2017 US Census data for San Diego County (one row per census tract) to:
1. Build a table of income distributions for each area (census tract).
2. Plot a choropleth map of the county using GeoPandas, coloring each area by its median household income on a **light red &rarr; dark red** scale.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import shapely
from shapely import wkt
import geopandas as gpd

import warnings
warnings.simplefilter(action='ignore', category=Warning)

pd.set_option('display.max_columns', None)

## 1. Load the census data

In [ ]:
csv_path = r'C:\Users\aksha\OneDrive\Desktop\DS3\SmartRoutes\data\san-diego-census-geopandas\data\San Diego.csv'
df = pd.read_csv(csv_path, delimiter=',', encoding='ISO-8859-2')
df.head()

## 2. Build the income-distribution table

Each row of the source data already represents one census tract (an "area") in San Diego County. We pull out the area name, GEOID, and the income-related columns. We also bucket each tract's median household income into income brackets so we have a true distribution view across the county.

In [ ]:
income_table = df[['GEOID', 'NAME', 'tract', 'total_pop', 'hh_total',
                   'median_hh_income', 'income_gini']].copy()

income_table['area'] = income_table['NAME'].str.replace(
    ', San Diego County, California', '', regex=False
)

income_brackets = [0, 25000, 50000, 75000, 100000, 150000, 200000, np.inf]
bracket_labels = ['<$25k', '$25k–$50k', '$50k–$75k', '$75k–$100k',
                  '$100k–$150k', '$150k–$200k', '$200k+']
income_table['income_bracket'] = pd.cut(
    income_table['median_hh_income'], bins=income_brackets,
    labels=bracket_labels, include_lowest=True
)

income_table = income_table[['GEOID', 'area', 'tract', 'total_pop', 'hh_total',
                             'median_hh_income', 'income_bracket', 'income_gini']]
income_table = income_table.sort_values('median_hh_income', ascending=False).reset_index(drop=True)
income_table.head(15)

In [ ]:
# Summary distribution across the whole county (how many tracts fall in each bracket)
bracket_summary = (income_table['income_bracket']
                   .value_counts()
                   .reindex(bracket_labels)
                   .rename('num_tracts')
                   .to_frame())
bracket_summary['pct_of_tracts'] = (
    100 * bracket_summary['num_tracts'] / bracket_summary['num_tracts'].sum()
).round(2)
bracket_summary

In [ ]:
# County-wide income summary statistics
income_table['median_hh_income'].describe().round(2)

## 3. Build the GeoDataFrame

The `geometry` column in the CSV is stored as Well-Known Text (WKT). We parse it into shapely geometries and assign the EPSG:26946 CRS (NAD83 / California zone 6) used by this dataset.

In [ ]:
gdf = gpd.GeoDataFrame(df.copy())
gdf['geometry'] = gdf['geometry'].apply(wkt.loads)
gdf = gdf.set_geometry('geometry')
gdf.crs = 'EPSG:26946'

gdf = gdf.merge(
    income_table[['GEOID', 'area', 'income_bracket']],
    on='GEOID', how='left'
)
gdf[['area', 'median_hh_income', 'income_bracket', 'geometry']].head()

## 4. Choropleth map: median household income (light red → dark red)

Matplotlib's built-in `Reds` colormap already ramps from a very light pink-red up to a deep, dark red, which matches the requested key.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 14))

gdf.plot(
    column='median_hh_income',
    cmap='Reds',
    linewidth=0.3,
    edgecolor='#666666',
    legend=True,
    legend_kwds={
        'label': 'Median household income (USD)',
        'orientation': 'vertical',
        'shrink': 0.6,
    },
    ax=ax,
)

ax.set_title('San Diego County — Median Household Income by Census Tract',
             fontsize=16, pad=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Optional: explicit light-red → dark-red ramp

If you want full control over the endpoints of the color key (e.g. a very pale red on the low end and an almost-black red on the high end) you can build a custom `LinearSegmentedColormap`:

In [ ]:
light_to_dark_red = LinearSegmentedColormap.from_list(
    'light_to_dark_red',
    ['#fff0f0', '#fcbba1', '#fb6a4a', '#cb181d', '#67000d'],
)

fig, ax = plt.subplots(1, 1, figsize=(14, 14))
gdf.plot(
    column='median_hh_income',
    cmap=light_to_dark_red,
    linewidth=0.3,
    edgecolor='#555555',
    legend=True,
    legend_kwds={
        'label': 'Median household income (USD)',
        'orientation': 'vertical',
        'shrink': 0.6,
    },
    missing_kwds={'color': 'lightgrey', 'label': 'No data'},
    ax=ax,
)
ax.set_title('San Diego County Income — Light Red (low) to Dark Red (high)',
             fontsize=16, pad=14)
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 5. Save the income-distribution table

Drop a CSV next to the existing outputs so the table can be reused by other notebooks.

In [ ]:
from pathlib import Path

output_dir = Path.cwd() / 'output_csvs'
output_dir.mkdir(parents=True, exist_ok=True)

income_table.to_csv(output_dir / 'san_diego_income_distribution.csv', index=False)
print('Saved:', output_dir / 'san_diego_income_distribution.csv')